# 11 — Mars process readiness audit

This is a gate, not a process-discovery notebook. It checks whether each proposed Mars process has the required measurements and validation evidence before Earth matching is allowed. It never creates fallback data or labels a cluster as a process.

In [ ]:
from pathlib import Path
import json
import pandas as pd

ROOT = Path('/kaggle/working') if Path('/kaggle/working').exists() else Path.cwd()
CONFIG = Path('/kaggle/input')
spec_path = next(iter(CONFIG.rglob('mars_process_specifications.json')), Path('configs/mars_process_specifications.json'))
specs = json.loads(spec_path.read_text(encoding='utf-8'))
candidate_files = list(ROOT.rglob('mars_candidate_validation_results.parquet'))
candidate = pd.read_parquet(candidate_files[0]) if candidate_files else None

rows = []
for spec in specs:
    required = list(spec.get('mars_evidence_variables', []))
    available = [] if candidate is None else [x for x in required if x in candidate.columns]
    dust_gate = spec['process_id'] == 'fine_particulate_dust_transport' and candidate is not None and 'validated_interpretation' in candidate.columns
    if spec.get('status') != 'supported_for_earth_screening':
        decision = 'inconclusive_pending_mars_evidence'
    elif not dust_gate:
        decision = 'blocked_missing_validation_output'
    else:
        decision = 'review_required_before_earth_matching'
    rows.append({'process_id': spec['process_id'], 'declared_status': spec.get('status'), 'required_evidence_count': len(required), 'available_named_columns': len(available), 'decision': decision})

readiness = pd.DataFrame(rows)
display(readiness)
OUT = ROOT / 'mars_process_readiness'
OUT.mkdir(exist_ok=True)
readiness.to_csv(OUT / 'process_readiness.csv', index=False)
(OUT / 'process_readiness.json').write_text(readiness.to_json(orient='records', indent=2), encoding='utf-8')
print('No process may be Earth-matched unless its decision is explicitly reviewed and supported.')